# 3. Label Encoding & Stratified Train–Validation Split

**Goal**: Convert human-readable labels into stable numerical IDs and create a fair 80-20 split that respects class imbalance.

In [12]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
import os

print("Libraries imported.")

Libraries imported.


## 2.1 Freeze the Label Space

In [13]:
file_path = '/Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/Primary_Emotions_Processed.xlsx'
df = pd.read_excel(file_path)

# Extract unique labels and sort alphabetically
unique_labels = sorted(df['Primary'].unique())
print(f"Total unique labels: {len(unique_labels)}")

# Create mapping
label_map = {label: idx for idx, label in enumerate(unique_labels)}

# Save mapping immediately
map_path = '/Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/label_map.json'
with open(map_path, 'w') as f:
    json.dump(label_map, f, indent=4)

print(f"Label map saved to {map_path}")
print("Sample mapping:")
print({k: label_map[k] for k in list(label_map)[:5]})

Total unique labels: 46
Label map saved to /Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/label_map.json
Sample mapping:
{'Anger': 0, 'Betrayal': 1, 'Calmness': 2, 'Care': 3, 'Caution': 4}


## 2.2 Encode Labels

In [14]:
df['label_id'] = df['Primary'].map(label_map)
display(df[['Primary', 'label_id']].head())

,Primary,label_id
0,Reverence,38
1,Reverence,38
2,Reverence,38
3,Reverence,38
4,Reverence,38


## 2.3 Verify Label Coverage

In [15]:
missing = df['label_id'].isnull().sum()
print(f"Rows with missing label_id: {missing}")

unique_ids = df['label_id'].nunique()
print(f"Total unique label_ids: {unique_ids}")

if missing == 0 and unique_ids == 46:
    print("✅ Label encoding verified.")
else:
    print("❌ ERROR: Label encoding failed!")

Rows with missing label_id: 0
Total unique label_ids: 46
✅ Label encoding verified.


## 2.4 Stratified Train–Validation Split

In [16]:
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label_id'], 
    random_state=42
)

print(f"Train shape: {train_df.shape}")
print(f"Val shape:   {val_df.shape}")

Train shape: (3299, 8)
Val shape:   (825, 8)


## 2.5 Validate the Split

In [17]:
# Total Integrity Check
total_samples = len(train_df) + len(val_df)
expected_samples = len(df)
print(f"Total Samples (Train + Val): {total_samples} / {expected_samples}")

# Label Distribution Check
train_labels = set(train_df['label_id'].unique())
val_labels = set(val_df['label_id'].unique())
all_labels = set(df['label_id'].unique())

missing_in_train = all_labels - train_labels
missing_in_val = all_labels - val_labels

print(f"Labels missing in Train: {missing_in_train}")
print(f"Labels missing in Val:   {missing_in_val}")

if len(missing_in_train) == 0 and len(missing_in_val) == 0:
    print("✅ Stratification successful. All labels present in both sets.")
else:
    print("⚠️ Warning: Some labels missing in split (likely extremely rare classes).")

Total Samples (Train + Val): 4124 / 4124
Labels missing in Train: set()
Labels missing in Val:   set()
✅ Stratification successful. All labels present in both sets.


## 2.6 Save Artifacts

In [18]:
train_path = '/Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/train.xlsx'
val_path = '/Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/val.xlsx'

train_df.to_excel(train_path, index=False)
val_df.to_excel(val_path, index=False)

print(f"Train set saved to {train_path}")
print(f"Validation set saved to {val_path}")
print("Step 2 Complete ✅")

Train set saved to /Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/train.xlsx
Validation set saved to /Users/hemishjain22/Desktop/Hackathons/hack4healtj/round1/val.xlsx
Step 2 Complete ✅
